# Hotel Review Classification — Teacher-Style Transformer (Kaggle)

This notebook solves the **same HRAST problem** as the original `kaggle_train(2).ipynb`, but replaces the two pretrained `bert-base-uncased` models with the encoder-only Transformer style used in `Lab5.ipynb`.

It keeps the teacher notebook's main modeling ideas:

- build our own vocabulary
- convert words to integer token IDs
- use `nn.Embedding`
- add sinusoidal positional encoding
- use `nn.TransformerEncoderLayer` + `nn.TransformerEncoder`
- create a padding mask
- mean-pool the encoder outputs
- use a final `nn.Linear` classifier
- write the PyTorch training loop directly

The task remains unchanged:

1. **Sentiment classification:** `negative`, `neutral`, `positive`
2. **Aspect classification:** 21 independent yes/no hotel-aspect labels

The cleaning, 70/15/15 split, validation/test metrics, best-checkpoint logic, aspect threshold search, and inference examples follow the current HRAST notebook.


## Step 0: Install packages and import libraries

`iterative-stratification` is only used to reproduce the same multilabel-stratified train/validation/test split as the current notebook. No Hugging Face Transformer model is downloaded.


In [ ]:
!pip install --quiet iterative-stratification==0.1.9


In [ ]:
import re
import json
import glob
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Using CPU")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
ARTIFACTS_DIR = WORKING_DIR / "artifacts_teacher_style"
SENTIMENT_DIR = ARTIFACTS_DIR / "sentiment"
ASPECT_DIR = ARTIFACTS_DIR / "aspects"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)
ASPECT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = glob.glob("/kaggle/input/**/HRAST.csv", recursive=True)
if len(csv_files) == 0:
    raise FileNotFoundError("HRAST.csv not found under /kaggle/input. Attach the HRAST dataset first.")

RAW_CSV_PATH = csv_files[0]
print("Found dataset at:", RAW_CSV_PATH)


## Step 1: Define the labels

This is the same label setup as the current notebook.


In [ ]:
SENTIMENT_COLUMNS = ["positive", "negative", "neutral"]
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
SENTIMENT_TO_ID = {"negative": 0, "neutral": 1, "positive": 2}

ASPECT_COLUMNS = [
    "Clean", "Comfort", "Facilities/Amenities", "Location",
    "Restaurant (dinner)", "Staff", "View (Balcony)", "Breakfast", "Room",
    "Pool", "Beach", "Bathroom/Shower (toilet)", "Bar", "Bed", "Parking",
    "Noise", "Reception-checkin", "Lift", "Value for money", "Wi-Fi", "Generic",
]

LABEL_COLUMNS = SENTIMENT_COLUMNS + ASPECT_COLUMNS


## Step 2: Load and clean HRAST

These cleaning steps are kept from the current notebook so the data problem does not change.


In [ ]:
df = pd.read_csv(RAW_CSV_PATH, encoding="utf-8-sig")
print("Rows in raw file:", len(df))

for column in df.columns:
    if str(column).startswith("Unnamed"):
        df = df.drop(columns=[column])


In [ ]:
def clean_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["review"] = df["review"].apply(clean_text)
df = df[df["review"] != ""]
print("Rows after removing empty reviews:", len(df))


In [ ]:
def row_labels_are_binary(row):
    for column in LABEL_COLUMNS:
        if row[column] not in (0, 1, "0", "1"):
            return False
    return True

keep_mask = df.apply(row_labels_are_binary, axis=1)
df = df[keep_mask]
print("Rows after removing non-binary label values:", len(df))


In [ ]:
for column in SENTIMENT_COLUMNS:
    df[column] = df[column].astype(int)
for column in ASPECT_COLUMNS:
    df[column] = df[column].astype(int)

sentiment_total = df["positive"] + df["negative"] + df["neutral"]
df = df[sentiment_total == 1]
print("Rows after keeping exactly one sentiment label:", len(df))


In [ ]:
def sentiment_name(row):
    if row["negative"] == 1:
        return "negative"
    if row["neutral"] == 1:
        return "neutral"
    return "positive"

df["sentiment"] = df.apply(sentiment_name, axis=1)


In [ ]:
rows_to_keep = []

for review_text, group in df.groupby("review"):
    first_row_labels = group[LABEL_COLUMNS].iloc[0].tolist()
    all_copies_agree = True

    for row_number in range(len(group)):
        this_row_labels = group[LABEL_COLUMNS].iloc[row_number].tolist()
        if this_row_labels != first_row_labels:
            all_copies_agree = False

    if all_copies_agree:
        rows_to_keep.append(group.iloc[0])

df = pd.DataFrame(rows_to_keep).reset_index(drop=True)
print("Final cleaned rows:", len(df))


## Step 3: Split into train / validation / test

Same 70/15/15 multilabel-stratified split as the current notebook.


In [ ]:
stratify_columns = ASPECT_COLUMNS + SENTIMENT_COLUMNS
label_matrix = df[stratify_columns].to_numpy()

splitter_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.30, random_state=SEED
)
train_index, rest_index = next(splitter_1.split(df, label_matrix))

train_df = df.iloc[train_index].reset_index(drop=True)
rest_df = df.iloc[rest_index].reset_index(drop=True)

rest_label_matrix = rest_df[stratify_columns].to_numpy()
splitter_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.50, random_state=SEED
)
validation_index, test_index = next(splitter_2.split(rest_df, rest_label_matrix))

validation_df = rest_df.iloc[validation_index].reset_index(drop=True)
test_df = rest_df.iloc[test_index].reset_index(drop=True)

print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

train_df.to_csv(DATA_DIR / "train.csv", index=False)
validation_df.to_csv(DATA_DIR / "validation.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)


## Step 4: Build our own vocabulary and encode reviews

This replaces BERT's pretrained tokenizer with the teacher notebook's vocabulary → integer IDs → padding approach.

The vocabulary is built from **training reviews only** so validation/test text does not leak into training. Unknown words become `<UNK>` and padding becomes `<PAD>`.


In [ ]:
MAX_LEN = 128

vocab = {"<PAD>": 0, "<UNK>": 1}

for review in train_df["review"]:
    for word in review.lower().split():
        if word not in vocab:
            vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))
print("Example entries:", list(vocab.items())[:20])


In [ ]:
def encode_review(review, max_len=MAX_LEN):
    tokens = []
    for word in review.lower().split():
        tokens.append(vocab.get(word, vocab["<UNK>"]))

    tokens = tokens[:max_len]
    tokens = tokens + [vocab["<PAD>"]] * (max_len - len(tokens))
    return tokens


def encode_text_list(texts):
    return torch.tensor([encode_review(text) for text in texts], dtype=torch.long)

X_train = encode_text_list(train_df["review"].tolist())
X_validation = encode_text_list(validation_df["review"].tolist())
X_test = encode_text_list(test_df["review"].tolist())

print("X_train shape:", X_train.shape)


In [ ]:
train_sentiment_labels = torch.tensor(
    [SENTIMENT_TO_ID[name] for name in train_df["sentiment"]],
    dtype=torch.long,
)
validation_sentiment_labels = torch.tensor(
    [SENTIMENT_TO_ID[name] for name in validation_df["sentiment"]],
    dtype=torch.long,
)
test_sentiment_labels = torch.tensor(
    [SENTIMENT_TO_ID[name] for name in test_df["sentiment"]],
    dtype=torch.long,
)

train_aspect_labels = torch.tensor(
    train_df[ASPECT_COLUMNS].to_numpy(), dtype=torch.float32
)
validation_aspect_labels = torch.tensor(
    validation_df[ASPECT_COLUMNS].to_numpy(), dtype=torch.float32
)
test_aspect_labels = torch.tensor(
    test_df[ASPECT_COLUMNS].to_numpy(), dtype=torch.float32
)

print("Sentiment label tensor:", train_sentiment_labels.shape)
print("Aspect label tensor:", train_aspect_labels.shape)


## Step 5: Teacher-style encoder-only Transformer

This model follows `Lab5.ipynb` closely. The only required change is `num_classes`: sentiment uses 3 outputs and aspects use 21 outputs.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        ).unsqueeze(0)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=64,
        nhead=2,
        num_layers=2,
        num_classes=1,
        max_len=128,
    ):
        super().__init__()
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            batch_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x, padding_mask=None):
        # 1. Word embeddings
        x = self.embedding(x) * math.sqrt(self.d_model)

        # 2. Positional encoding
        x = self.pos_encoder(x)

        # 3. Transformer encoder
        encoded = self.transformer_encoder(
            x,
            src_key_padding_mask=padding_mask,
        )

        # 4. Mean pooling, matching the teacher notebook
        pooled = encoded.mean(dim=1)

        # 5. Final classifier
        logits = self.classifier(pooled)
        return logits


## Step 6: Training settings and DataLoaders

The teacher notebook uses `DataLoader` + `TensorDataset`, so this notebook does the same.

Because this Transformer is trained from random initialization rather than fine-tuning pretrained BERT weights, its learning rate is much larger than the BERT notebook's `2e-5`.


In [ ]:
EPOCHS = 10
LEARNING_RATE = 0.001
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64

sentiment_train_loader = DataLoader(
    TensorDataset(X_train, train_sentiment_labels),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
)
sentiment_validation_loader = DataLoader(
    TensorDataset(X_validation, validation_sentiment_labels),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)
sentiment_test_loader = DataLoader(
    TensorDataset(X_test, test_sentiment_labels),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)

aspect_train_loader = DataLoader(
    TensorDataset(X_train, train_aspect_labels),
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
)
aspect_validation_loader = DataLoader(
    TensorDataset(X_validation, validation_aspect_labels),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)
aspect_test_loader = DataLoader(
    TensorDataset(X_test, test_aspect_labels),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)


## Step 7: Sentiment model

For 3-class sentiment classification, the same teacher-style Transformer produces 3 logits. `CrossEntropyLoss` compares those logits with one class ID per review.


In [ ]:
sentiment_model = TransformerClassifier(
    vocab_size=len(vocab),
    d_model=64,
    nhead=2,
    num_layers=2,
    num_classes=3,
    max_len=MAX_LEN,
).to(device)

sentiment_optimizer = optim.Adam(sentiment_model.parameters(), lr=LEARNING_RATE)
sentiment_loss_function = nn.CrossEntropyLoss()

print(sentiment_model)


In [ ]:
def train_sentiment_one_epoch():
    sentiment_model.train()
    total_loss = 0.0

    for inputs, targets in sentiment_train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        padding_mask = (inputs == vocab["<PAD>"]).to(device)

        sentiment_optimizer.zero_grad()
        logits = sentiment_model(inputs, padding_mask=padding_mask)
        loss = sentiment_loss_function(logits, targets)
        loss.backward()
        sentiment_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(sentiment_train_loader)


def evaluate_sentiment(loader):
    sentiment_model.eval()
    all_predictions = []
    all_true_labels = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            padding_mask = (inputs == vocab["<PAD>"]).to(device)

            logits = sentiment_model(inputs, padding_mask=padding_mask)
            predictions = torch.argmax(logits, dim=1).cpu().tolist()

            all_predictions.extend(predictions)
            all_true_labels.extend(targets.tolist())

    accuracy = accuracy_score(all_true_labels, all_predictions)
    macro_f1 = f1_score(all_true_labels, all_predictions, average="macro")
    return accuracy, macro_f1


In [ ]:
best_sentiment_macro_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_sentiment_one_epoch()
    val_accuracy, val_macro_f1 = evaluate_sentiment(sentiment_validation_loader)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss = {train_loss:.4f} | "
        f"validation accuracy = {val_accuracy:.4f} | "
        f"validation macro-F1 = {val_macro_f1:.4f}"
    )

    if val_macro_f1 > best_sentiment_macro_f1:
        best_sentiment_macro_f1 = val_macro_f1
        torch.save(sentiment_model.state_dict(), SENTIMENT_DIR / "model.pt")
        print("  New best sentiment model saved.")


In [ ]:
sentiment_model.load_state_dict(
    torch.load(SENTIMENT_DIR / "model.pt", map_location=device)
)

test_accuracy, test_macro_f1 = evaluate_sentiment(sentiment_test_loader)
print(f"Sentiment test accuracy = {test_accuracy:.4f}")
print(f"Sentiment test macro-F1 = {test_macro_f1:.4f}")


## Step 8: Aspect model

The architecture is the same, but now the classifier has 21 outputs. Each output is an independent yes/no decision, so training uses `BCEWithLogitsLoss`.


In [ ]:
aspect_model = TransformerClassifier(
    vocab_size=len(vocab),
    d_model=64,
    nhead=2,
    num_layers=2,
    num_classes=len(ASPECT_COLUMNS),
    max_len=MAX_LEN,
).to(device)

aspect_optimizer = optim.Adam(aspect_model.parameters(), lr=LEARNING_RATE)
aspect_loss_function = nn.BCEWithLogitsLoss()


In [ ]:
def train_aspect_one_epoch():
    aspect_model.train()
    total_loss = 0.0

    for inputs, targets in aspect_train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        padding_mask = (inputs == vocab["<PAD>"]).to(device)

        aspect_optimizer.zero_grad()
        logits = aspect_model(inputs, padding_mask=padding_mask)
        loss = aspect_loss_function(logits, targets)
        loss.backward()
        aspect_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(aspect_train_loader)


def evaluate_aspects(loader, threshold=0.5):
    aspect_model.eval()
    all_predictions = []
    all_true_labels = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(device)
            padding_mask = (inputs == vocab["<PAD>"]).to(device)

            logits = aspect_model(inputs, padding_mask=padding_mask)
            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= threshold).int().cpu().numpy()

            all_predictions.extend(predictions.tolist())
            all_true_labels.extend(targets.int().numpy().tolist())

    macro_f1 = f1_score(
        all_true_labels,
        all_predictions,
        average="macro",
        zero_division=0,
    )
    micro_f1 = f1_score(
        all_true_labels,
        all_predictions,
        average="micro",
        zero_division=0,
    )
    return macro_f1, micro_f1


In [ ]:
best_aspect_macro_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_aspect_one_epoch()
    val_macro_f1, val_micro_f1 = evaluate_aspects(
        aspect_validation_loader,
        threshold=0.5,
    )

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss = {train_loss:.4f} | "
        f"validation macro-F1 = {val_macro_f1:.4f} | "
        f"validation micro-F1 = {val_micro_f1:.4f}"
    )

    if val_macro_f1 > best_aspect_macro_f1:
        best_aspect_macro_f1 = val_macro_f1
        torch.save(aspect_model.state_dict(), ASPECT_DIR / "model.pt")
        print("  New best aspect model saved.")


In [ ]:
aspect_model.load_state_dict(
    torch.load(ASPECT_DIR / "model.pt", map_location=device)
)

test_macro_f1, test_micro_f1 = evaluate_aspects(
    aspect_test_loader,
    threshold=0.5,
)
print(f"Aspect test macro-F1 at 0.50 = {test_macro_f1:.4f}")
print(f"Aspect test micro-F1 at 0.50 = {test_micro_f1:.4f}")


## Step 9: Tune the global aspect threshold

This is kept from the current notebook. The model weights are fixed; only the probability cutoff used to turn each sigmoid output into 0/1 is selected on the validation set.


In [ ]:
best_threshold = 0.5
best_threshold_macro_f1 = -1.0

threshold = 0.20
while threshold <= 0.80 + 1e-9:
    current_threshold = round(threshold, 2)
    macro_f1, micro_f1 = evaluate_aspects(
        aspect_validation_loader,
        threshold=current_threshold,
    )

    print(
        f"threshold = {current_threshold:.2f} | "
        f"validation macro-F1 = {macro_f1:.4f} | "
        f"validation micro-F1 = {micro_f1:.4f}"
    )

    if macro_f1 > best_threshold_macro_f1:
        best_threshold_macro_f1 = macro_f1
        best_threshold = current_threshold

    threshold += 0.05

print("Best threshold:", best_threshold)


In [ ]:
test_macro_f1, test_micro_f1 = evaluate_aspects(
    aspect_test_loader,
    threshold=best_threshold,
)
print(f"Aspect final test macro-F1 = {test_macro_f1:.4f}")
print(f"Aspect final test micro-F1 = {test_micro_f1:.4f}")


## Step 10: Save vocabulary and model settings

Unlike Hugging Face's `save_pretrained`, a custom PyTorch model needs its vocabulary and architecture settings saved separately so it can be reconstructed later.


In [ ]:
with open(ARTIFACTS_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

config = {
    "max_len": MAX_LEN,
    "d_model": 64,
    "nhead": 2,
    "num_layers": 2,
    "sentiment_labels": SENTIMENT_LABELS,
    "aspect_columns": ASPECT_COLUMNS,
    "aspect_threshold": best_threshold,
}

with open(ARTIFACTS_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved artifacts to:", ARTIFACTS_DIR)


## Step 11: Inference

Inference uses the same manually built vocabulary, padding mask, Transformer encoder, and task-specific output interpretation.


In [ ]:
def prepare_single_text(text):
    encoded = encode_review(clean_text(text))
    input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
    padding_mask = (input_tensor == vocab["<PAD>"]).to(device)
    return input_tensor, padding_mask


def predict_sentiment(text):
    sentiment_model.eval()
    input_tensor, padding_mask = prepare_single_text(text)

    with torch.no_grad():
        logits = sentiment_model(input_tensor, padding_mask=padding_mask)
        probabilities = torch.softmax(logits, dim=1)[0].cpu().tolist()

    scores = {}
    for i in range(len(SENTIMENT_LABELS)):
        scores[SENTIMENT_LABELS[i]] = round(probabilities[i], 4)

    predicted_id = int(torch.argmax(logits, dim=1)[0])
    predicted_label = SENTIMENT_LABELS[predicted_id]
    return predicted_label, scores


In [ ]:
def predict_aspects(text):
    aspect_model.eval()
    input_tensor, padding_mask = prepare_single_text(text)

    with torch.no_grad():
        logits = aspect_model(input_tensor, padding_mask=padding_mask)
        probabilities = torch.sigmoid(logits)[0].cpu().tolist()

    detected_aspects = []
    for i in range(len(ASPECT_COLUMNS)):
        if probabilities[i] >= best_threshold:
            detected_aspects.append(
                (ASPECT_COLUMNS[i], round(probabilities[i], 4))
            )

    detected_aspects.sort(key=lambda item: item[1], reverse=True)
    return detected_aspects


In [ ]:
example_sentences = [
    "The staff were friendly and the room was spotless.",
    "The Wi-Fi was slow and the room was noisy.",
    "Breakfast was poor but the staff were excellent.",
    "The hotel is located three kilometres from the airport.",
]

for sentence in example_sentences:
    predicted_label, scores = predict_sentiment(sentence)
    aspects_found = predict_aspects(sentence)

    print("Review:", sentence)
    print("Sentiment:", predicted_label, scores)
    print("Aspects:", aspects_found)
    print()


## What changed compared with the BERT notebook?

The **problem did not change**. Only the model implementation changed.

| Part | Current notebook | This notebook |
|---|---|---|
| Tokenization | pretrained BERT tokenizer | own word vocabulary + `<UNK>` + `<PAD>` |
| Embeddings | pretrained BERT embeddings | `nn.Embedding` learned from scratch |
| Position information | inside BERT | sinusoidal `PositionalEncoding` |
| Encoder | pretrained BERT encoder | `nn.TransformerEncoder` from random initialization |
| Pooling/classification | BERT sequence-classification head | mean pooling + `nn.Linear` |
| Sentiment output | 3 logits | 3 logits |
| Aspect output | 21 logits | 21 logits |
| Sentiment loss | CrossEntropy | CrossEntropy |
| Aspect loss | BCEWithLogits | BCEWithLogits |
| Dataset/split/metrics | HRAST, 70/15/15, F1 | same |

Because this version trains the language representation from scratch on HRAST instead of starting from pretrained BERT, it should not be expected to match BERT's accuracy automatically. Its main advantage for this lab is that it follows the teacher's Transformer construction much more closely and makes each major Transformer component visible in the notebook.
